In [2]:
import httpx, json
from openai import OpenAI

def log_request(request: httpx.Request):
    try:
        print("\n=== OUTBOUND REQUEST ===")
        print(request.method, request.url)
        if request.content:
            try:
                print(json.dumps(json.loads(request.content), indent=2))
            except Exception:
                print(request.content.decode(errors="ignore")[:2000])
    except Exception as e:
        print(f"[log_request] suppressed error: {e!r}")

def log_response(response: httpx.Response):
    try:
        print("\n=== INBOUND RESPONSE ===")
        print(response.status_code, response.reason_phrase)

        # Ensure the streaming body is fully loaded into memory.
        response.read()

        ctype = response.headers.get("content-type", "")
        if "application/json" in ctype.lower():
            try:
                print(json.dumps(response.json(), indent=2))
            except Exception:
                print(response.text[:2000])
        else:
            print(response.text[:2000])
    except Exception as e:
        # Never let the hook raise; that will break the client call.
        print(f"[log_response] suppressed error: {e!r}")

http_client = httpx.Client(event_hooks={"request": [log_request], "response": [log_response]})

client = OpenAI(
    base_url="http://localhost:11434/v1",  # Ollama OpenAI-compat endpoint
    api_key="ollama",
    http_client=http_client,
)


In [ ]:
from pydantic import BaseModel, Field
from typing import List, Dict

class ReviewClassification(BaseModel):
    """
    You are a expert customer testimonial reviewer who loves to read a customer's review for a given product to identify topics. \n Your task is to read a given review: classify the topic and rate the sentiment
    """
    classification: str = Field(..., description = "Topic of the review")
    classification_explanation: str = Field(..., description = "explanation of why the classification rating was given")
    sentiment: int = Field(..., description = "1-5 rating (1 very negative to 5 very positive) for the topic")
    sentiment_explanation: str =  Field(..., description = "explanation of why the sentiment rating was given")



In [9]:
customer_review = "As the title says, this is simply the best backpack I've ever used. It's extremely comfortable and sometimes I feel like I forgot to pack my laptop and lunch it's that light. \r\n\r\nMy friends think it's a waste of money (they're not totally wrong) but even when they tried it on they agreed that its simply better than anything they've used. \r\n\r\nI'm a small male who's about 169cm ±2cm and very skinny at 45kg, it's a tad big for me but still fits quite nicely when pulling the adjustable shoulder straps nearly all the way in. \r\n\r\nIt fits exactly a framework laptop, lunch box, pencil case, glasses case, work uniform, whole old backpack, 40oz LTT water bottle, keys and other small things in the front pouch and side pocket. \r\n\r\nThe verdict: if you're just as, if not smaller than me I'd highly recommend waiting for the Yvonne sized one as Linus mentioned a wan or 3 ago (and showed off in the FP exclusive BTS of the AMD upgrade thing Yvonne was in recently) but if you value products that are going to last a long while and you like LTT there simply isn't a better bag for the (albeit quite high) price. (PS Linus I know you read these sometimes and in which case hello!)"

In [10]:
final_prompt = f"Input: [{customer_review}]"

In [11]:
json_completion = client.beta.chat.completions.parse(
    model="gemma3:12b-it-q8_0",   
    response_format= ReviewClassification,
    messages=[
            {"role": "system", "content": final_prompt},
            # {"role": "user", "content": thought_extraction},
    ],
    temperature=0.2,
    )
ReviewClassification.model_validate(json_completion.choices[0].message.parsed)
json_completion.choices[0].message.parsed.model_dump()


=== OUTBOUND REQUEST ===
POST http://localhost:11434/v1/chat/completions
{
  "messages": [
    {
      "role": "system",
      "content": "Input: [As the title says, this is simply the best backpack I've ever used. It's extremely comfortable and sometimes I feel like I forgot to pack my laptop and lunch it's that light. \r\n\r\nMy friends think it's a waste of money (they're not totally wrong) but even when they tried it on they agreed that its simply better than anything they've used. \r\n\r\nI'm a small male who's about 169cm \u00b12cm and very skinny at 45kg, it's a tad big for me but still fits quite nicely when pulling the adjustable shoulder straps nearly all the way in. \r\n\r\nIt fits exactly a framework laptop, lunch box, pencil case, glasses case, work uniform, whole old backpack, 40oz LTT water bottle, keys and other small things in the front pouch and side pocket. \r\n\r\nThe verdict: if you're just as, if not smaller than me I'd highly recommend waiting for the Yvonne siz

{'classification': 'Positive',
 'classification_explanation': 'The reviewer consistently praises the backpack, describing it as "simply the best," "extremely comfortable," and better than anything they\'ve used. While acknowledging the high price and suggesting a smaller size for those smaller than them, the overall sentiment is overwhelmingly positive.',
 'sentiment': 0,
 'sentiment_explanation': "The overall sentiment is positive, with strong positive language used throughout the review. The minor caveats (price, size for smaller individuals) don't detract from the core message of satisfaction and recommendation. The direct address to Linus further reinforces a positive and engaged tone."}

In [12]:
ReviewClassification.model_json_schema()

{'description': "   You are a expert customer testimonial reviewer who loves to read a customer's review for a given product to identify topics. \nYour task is to read a given review: classify the topic and rate the sentiment\n   ",
 'properties': {'classification': {'description': 'Topic of the review',
   'title': 'Classification',
   'type': 'string'},
  'classification_explanation': {'description': 'explanation of why the classification rating was given',
   'title': 'Classification Explanation',
   'type': 'string'},
  'sentiment': {'description': '1-5 rating (1 very negative to 5 very positive) for the topic',
   'title': 'Sentiment',
   'type': 'integer'},
  'sentiment_explanation': {'description': 'explanation of why the sentiment rating was given',
   'title': 'Sentiment Explanation',
   'type': 'string'}},
 'required': ['classification',
  'classification_explanation',
  'sentiment',
  'sentiment_explanation'],
 'title': 'ReviewClassification',
 'type': 'object'}

In [13]:
json_completion = client.chat.completions.create(
    model="gemma3:12b-it-q8_0",   
    # response_format= ReviewClassification,
    messages=[
            {"role": "system", "content": str(ReviewClassification.model_json_schema())},
            {"role": "system", "content": final_prompt},
            # {"role": "user", "content": thought_extraction},
    ],
    temperature=0.2,
    )
# ReviewClassification.model_validate(json_completion.choices[0].message.parsed)
# json_completion.choices[0].message.parsed.model_dump()


=== OUTBOUND REQUEST ===
POST http://localhost:11434/v1/chat/completions
{
  "messages": [
    {
      "role": "system",
      "content": "{'description': \"   You are a expert customer testimonial reviewer who loves to read a customer's review for a given product to identify topics. \\nYour task is to read a given review: classify the topic and rate the sentiment\\n   \", 'properties': {'classification': {'description': 'Topic of the review', 'title': 'Classification', 'type': 'string'}, 'classification_explanation': {'description': 'explanation of why the classification rating was given', 'title': 'Classification Explanation', 'type': 'string'}, 'sentiment': {'description': '1-5 rating (1 very negative to 5 very positive) for the topic', 'title': 'Sentiment', 'type': 'integer'}, 'sentiment_explanation': {'description': 'explanation of why the sentiment rating was given', 'title': 'Sentiment Explanation', 'type': 'string'}}, 'required': ['classification', 'classification_explanation'

In [14]:
json_completion.choices[0].message.content

'```json\n{\n  "classification": "Product Quality & Size",\n  "classification_explanation": "The review focuses heavily on the backpack\'s comfort, durability, and size/fit. It discusses the quality of the product and how it compares to others, as well as the reviewer\'s experience with its size relative to their body type.",\n  "sentiment": 5,\n  "sentiment_explanation": "The reviewer expresses overwhelmingly positive feelings about the backpack. They call it \'the best backpack I\'ve ever used,\' praise its comfort and durability, and recommend it despite acknowledging it might be too large for some. The enthusiastic tone and strong recommendation indicate a very positive sentiment."\n}\n```'

In [17]:
import re
pattern = re.compile(r"```json\s*([\s\S]*?)\s*```", re.IGNORECASE)
text = re.sub(pattern, r"\1", json_completion.choices[0].message.content)
json.loads(text)

{'classification': 'Product Quality & Size',
 'classification_explanation': "The review focuses heavily on the backpack's comfort, durability, and size/fit. It discusses the quality of the product and how it compares to others, as well as the reviewer's experience with its size relative to their body type.",
 'sentiment': 5,
 'sentiment_explanation': "The reviewer expresses overwhelmingly positive feelings about the backpack. They call it 'the best backpack I've ever used,' praise its comfort and durability, and recommend it despite acknowledging it might be too large for some. The enthusiastic tone and strong recommendation indicate a very positive sentiment."}